<a href="https://colab.research.google.com/github/AishvaryaGovindaraju/DL-XAI/blob/main/DL_XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [ ]:
from pathlib import Path

folders = [
    "project/data/raw",
    "project/data/processed",
    "project/src/preprocessing",
    "project/src/models",
    "project/src/xai",
    "project/src/evaluation",
    "project/src/statistics",
    "project/notebooks",
    "project/results/xai_outputs",
    "project/figures",
    "project/paper"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folder structure created successfully.")

Project folder structure created successfully.


In [ ]:
!pip install -q torch shap lime dice-ml xgboost scikit-learn pandas numpy scipy scikit-posthocs pingouin matplotlib seaborn kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 21.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 23.8 MB/s eta 0:00:00


In [ ]:
import torch
import shap
import lime
import dice_ml
import xgboost
import sklearn
import pandas
import numpy
import scipy
import scikit_posthocs
import pingouin
import matplotlib
import seaborn

print("All required libraries imported successfully.")
print("PyTorch version:", torch.__version__)

All required libraries imported successfully.
PyTorch version: 2.11.0+cpu


In [ ]:
import shutil
from pathlib import Path

# Create the required folder
Path("project/data/raw").mkdir(parents=True, exist_ok=True)

# Change this filename if Colab shows a slightly different uploaded name
source = "/content/diabetes_binary_health_indicators_BRFSS2015.csv"
destination = "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"

shutil.copy(source, destination)

print("Dataset copied successfully!")
print("Location:", destination)

Dataset copied successfully!
Location: project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv


In [ ]:
import pandas as pd

file_path = "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (253680, 22)

Columns:
['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


In [ ]:
# Check the target distribution

target_counts = df["Diabetes_binary"].value_counts().sort_index()
target_percent = df["Diabetes_binary"].value_counts(normalize=True).sort_index() * 100

print("Diabetes_binary distribution:")
print(target_counts)

print("\nPercentage distribution:")
print(target_percent)

print("\nClass labels:")
print("0 = No diabetes")
print("1 = Diabetes")

Diabetes_binary distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Percentage distribution:
Diabetes_binary
0.0    86.066698
1.0    13.933302
Name: proportion, dtype: float64

Class labels:
0 = No diabetes
1 = Diabetes


In [ ]:
# Check for missing values in every column

missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values in dataset:", missing_values.sum())

if missing_values.sum() == 0:
    print("✓ No missing values found.")
else:
    print("⚠ Missing values found.")

Missing values per column:
Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Total missing values in dataset: 0
✓ No missing values found.


In [ ]:
# Check data types and number of unique values

print("DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n\nUNIQUE VALUES PER COLUMN")
print("=" * 50)
print(df.nunique().sort_values())

DATA TYPES
Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


UNIQUE VALUES PER COLUMN
Diabetes_binary          2
HighBP                   2
HighChol                 2
CholCheck                2
Smoker                   2
Stroke                   2
HeartDiseaseorAttack     2
PhysActivity             2
AnyHealthcare            2
F

In [ ]:
# Inspect the actual unique values of important features

features_to_check = [
    "HighBP",
    "Sex",
    "GenHlth",
    "Age",
    "Education",
    "Income",
    "BMI",
    "MentHlth",
    "PhysHlth"
]

for feature in features_to_check:
    print(f"\n{feature}")
    print("-" * 40)
    print(sorted(df[feature].unique()))


HighBP
----------------------------------------
[np.float64(0.0), np.float64(1.0)]

Sex
----------------------------------------
[np.float64(0.0), np.float64(1.0)]

GenHlth
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

Age
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0)]

Education
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]

Income
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0)]

BMI
----------------------------------------
[np.float64(12.0), np.float64(13.0), np.

In [ ]:
# Descriptive statistics for all columns

pd.set_option("display.max_columns", None)

print("DESCRIPTIVE STATISTICS")
print("=" * 80)

print(df.describe().T)

DESCRIPTIVE STATISTICS
                         count       mean       std   min   25%   50%   75%  \
Diabetes_binary       253680.0   0.139333  0.346294   0.0   0.0   0.0   0.0   
HighBP                253680.0   0.429001  0.494934   0.0   0.0   0.0   1.0   
HighChol              253680.0   0.424121  0.494210   0.0   0.0   0.0   1.0   
CholCheck             253680.0   0.962670  0.189571   0.0   1.0   1.0   1.0   
BMI                   253680.0  28.382364  6.608694  12.0  24.0  27.0  31.0   
Smoker                253680.0   0.443169  0.496761   0.0   0.0   0.0   1.0   
Stroke                253680.0   0.040571  0.197294   0.0   0.0   0.0   0.0   
HeartDiseaseorAttack  253680.0   0.094186  0.292087   0.0   0.0   0.0   0.0   
PhysActivity          253680.0   0.756544  0.429169   0.0   1.0   1.0   1.0   
Fruits                253680.0   0.634256  0.481639   0.0   0.0   1.0   1.0   
Veggies               253680.0   0.811420  0.391175   0.0   1.0   1.0   1.0   
HvyAlcoholConsump     253680.

In [ ]:
# Correlation of each feature with the diabetes target

correlations = df.corr(numeric_only=True)["Diabetes_binary"].drop("Diabetes_binary")

correlations = correlations.sort_values(ascending=False)

print("Feature correlation with Diabetes_binary:")
print("=" * 60)
print(correlations)

Feature correlation with Diabetes_binary:
GenHlth                 0.293569
HighBP                  0.263129
DiffWalk                0.218344
BMI                     0.216843
HighChol                0.200276
Age                     0.177442
HeartDiseaseorAttack    0.177282
PhysHlth                0.171337
Stroke                  0.105816
MentHlth                0.069315
CholCheck               0.064761
Smoker                  0.060789
NoDocbcCost             0.031433
Sex                     0.031430
AnyHealthcare           0.016255
Fruits                 -0.040779
Veggies                -0.056584
HvyAlcoholConsump      -0.057056
PhysActivity           -0.118133
Education              -0.124456
Income                 -0.163919
Name: Diabetes_binary, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: divide the 30% temporary data equally
# This gives 15% validation and 15% test overall
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Dataset split completed.\n")

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Dataset split completed.

Training set: (177576, 21) (177576,)
Validation set: (38052, 21) (38052,)
Test set: (38052, 21) (38052,)


In [ ]:
stratify=y

In [ ]:
# Verify class distribution in each split

def show_distribution(name, target):
    counts = target.value_counts().sort_index()
    percentages = target.value_counts(normalize=True).sort_index() * 100

    print(f"\n{name}")
    print("-" * 50)

    for class_value in counts.index:
        print(
            f"Class {int(class_value)}: "
            f"{counts[class_value]:,} samples "
            f"({percentages[class_value]:.2f}%)"
        )

show_distribution("TRAINING SET", y_train)
show_distribution("VALIDATION SET", y_val)
show_distribution("TEST SET", y_test)


TRAINING SET
--------------------------------------------------
Class 0: 152,834 samples (86.07%)
Class 1: 24,742 samples (13.93%)

VALIDATION SET
--------------------------------------------------
Class 0: 32,750 samples (86.07%)
Class 1: 5,302 samples (13.93%)

TEST SET
--------------------------------------------------
Class 0: 32,750 samples (86.07%)
Class 1: 5,302 samples (13.93%)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Features specified by the project for standardization
scaled_features = [
    "BMI",
    "MentHlth",
    "PhysHlth",
    "Age"
]

# Create the scaler
scaler = StandardScaler()

# Fit ONLY on the training data
scaler.fit(X_train[scaled_features])

print("Scaler fitted successfully.")
print("\nFeatures being standardized:")
print(scaled_features)

print("\nTraining means learned by scaler:")
print(scaler.mean_)

print("\nTraining standard deviations learned by scaler:")
print(scaler.scale_)

Scaler fitted successfully.

Features being standardized:
['BMI', 'MentHlth', 'PhysHlth', 'Age']

Training means learned by scaler:
[28.37629522  3.19317926  4.24358021  8.03167095]

Training standard deviations learned by scaler:
[6.60785624 7.42707453 8.71942756 3.05035369]


In [ ]:
scaler.fit(X_train[scaled_features])

StandardScaler()

In [ ]:
# Create copies so the original split data remains unchanged
X_train_processed = X_train.copy()
X_val_processed = X_val.copy()
X_test_processed = X_test.copy()

# Apply the training-fitted scaler to all three datasets
X_train_processed[scaled_features] = scaler.transform(
    X_train[scaled_features]
)

X_val_processed[scaled_features] = scaler.transform(
    X_val[scaled_features]
)

X_test_processed[scaled_features] = scaler.transform(
    X_test[scaled_features]
)

print("Preprocessing transformation completed successfully.")
print("\nTraining shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Test shape:", X_test_processed.shape)

Preprocessing transformation completed successfully.

Training shape: (177576, 21)
Validation shape: (38052, 21)
Test shape: (38052, 21)


In [ ]:
# Verify that the training data was standardized correctly

print("TRAINING SET AFTER STANDARDIZATION")
print("=" * 60)

for feature in scaled_features:
    print(
        f"{feature:10s} | "
        f"mean = {X_train_processed[feature].mean():.6f} | "
        f"std = {X_train_processed[feature].std():.6f}"
    )

TRAINING SET AFTER STANDARDIZATION
BMI        | mean = 0.000000 | std = 1.000003
MentHlth   | mean = -0.000000 | std = 1.000003
PhysHlth   | mean = -0.000000 | std = 1.000003
Age        | mean = 0.000000 | std = 1.000003


In [ ]:
import joblib
from pathlib import Path

# Create model/preprocessing directory
Path("project/src/models").mkdir(parents=True, exist_ok=True)

# Save the fitted scaler
scaler_path = "project/src/models/scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Scaler saved successfully!")
print("Saved to:", scaler_path)

Scaler saved successfully!
Saved to: project/src/models/scaler.pkl


In [ ]:
# Verify that the saved scaler can be loaded

loaded_scaler = joblib.load(scaler_path)

print("Saved scaler loaded successfully!")
print("Scaled features:", loaded_scaler.feature_names_in_)

Saved scaler loaded successfully!
Scaled features: ['BMI' 'MentHlth' 'PhysHlth' 'Age']


In [ ]:
import numpy as np

# Convert target labels to float32
y_train_processed = y_train.to_numpy(dtype=np.float32)
y_val_processed = y_val.to_numpy(dtype=np.float32)
y_test_processed = y_test.to_numpy(dtype=np.float32)

print("Labels prepared successfully.")

print("\nTraining labels:")
print("Shape:", y_train_processed.shape)
print("Data type:", y_train_processed.dtype)
print("Unique values:", np.unique(y_train_processed))

print("\nValidation labels:")
print("Shape:", y_val_processed.shape)
print("Data type:", y_val_processed.dtype)
print("Unique values:", np.unique(y_val_processed))

print("\nTest labels:")
print("Shape:", y_test_processed.shape)
print("Data type:", y_test_processed.dtype)
print("Unique values:", np.unique(y_test_processed))

Labels prepared successfully.

Training labels:
Shape: (177576,)
Data type: float32
Unique values: [0. 1.]

Validation labels:
Shape: (38052,)
Data type: float32
Unique values: [0. 1.]

Test labels:
Shape: (38052,)
Data type: float32
Unique values: [0. 1.]


In [ ]:
# Convert processed feature DataFrames to NumPy arrays

X_train_np = X_train_processed.to_numpy(dtype=np.float32)
X_val_np = X_val_processed.to_numpy(dtype=np.float32)
X_test_np = X_test_processed.to_numpy(dtype=np.float32)

print("Feature arrays created successfully.")

print("\nTraining:")
print("Shape:", X_train_np.shape)
print("Data type:", X_train_np.dtype)

print("\nValidation:")
print("Shape:", X_val_np.shape)
print("Data type:", X_val_np.dtype)

print("\nTest:")
print("Shape:", X_test_np.shape)
print("Data type:", X_test_np.dtype)

Feature arrays created successfully.

Training:
Shape: (177576, 21)
Data type: float32

Validation:
Shape: (38052, 21)
Data type: float32

Test:
Shape: (38052, 21)
Data type: float32


In [ ]:
# Final preprocessing sanity check

print("FINAL PREPROCESSING SANITY CHECK")
print("=" * 60)

# Check NaN values
print("\nNaN values:")
print("Train:", np.isnan(X_train_np).sum())
print("Validation:", np.isnan(X_val_np).sum())
print("Test:", np.isnan(X_test_np).sum())

# Check infinite values
print("\nInfinite values:")
print("Train:", np.isinf(X_train_np).sum())
print("Validation:", np.isinf(X_val_np).sum())
print("Test:", np.isinf(X_test_np).sum())

# Check overall numerical ranges
print("\nOverall feature ranges:")
print("Train min:", X_train_np.min())
print("Train max:", X_train_np.max())
print("Validation min:", X_val_np.min())
print("Validation max:", X_val_np.max())
print("Test min:", X_test_np.min())
print("Test max:", X_test_np.max())

FINAL PREPROCESSING SANITY CHECK

NaN values:
Train: 0
Validation: 0
Test: 0

Infinite values:
Train: 0
Validation: 0
Test: 0

Overall feature ranges:
Train min: -2.4783068
Train max: 10.536504
Validation min: -2.4783068
Validation max: 10.0824995
Test min: -2.4783068
Test max: 10.536504


In [ ]:
import torch

# Convert feature arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)

# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train_processed, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_processed, dtype=torch.float32)

print("PyTorch tensors created successfully.")

print("\nTraining:")
print("X:", X_train_tensor.shape, X_train_tensor.dtype)
print("y:", y_train_tensor.shape, y_train_tensor.dtype)

print("\nValidation:")
print("X:", X_val_tensor.shape, X_val_tensor.dtype)
print("y:", y_val_tensor.shape, y_val_tensor.dtype)

print("\nTest:")
print("X:", X_test_tensor.shape, X_test_tensor.dtype)
print("y:", y_test_tensor.shape, y_test_tensor.dtype)

PyTorch tensors created successfully.

Training:
X: torch.Size([177576, 21]) torch.float32
y: torch.Size([177576]) torch.float32

Validation:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32

Test:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32


In [ ]:
import torch.nn as nn

class DiabetesDNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(21, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Second hidden layer
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Third hidden layer
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output layer
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Create the DNN model
model = DiabetesDNN()

print(model)

DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [ ]:
# Count trainable parameters

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

print("\nParameters by layer:")
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(f"{name:30s} {parameter.numel():,}")

Trainable parameters: 13185

Parameters by layer:
network.0.weight               2,688
network.0.bias                 128
network.3.weight               8,192
network.3.bias                 64
network.6.weight               2,048
network.6.bias                 32
network.9.weight               32
network.9.bias                 1


In [ ]:
import numpy as np

# Count classes in the training set
class_counts = np.bincount(y_train_processed.astype(int))

# Calculate balanced class weights
num_samples = len(y_train_processed)
num_classes = len(class_counts)

class_weights = num_samples / (num_classes * class_counts)

print("Training class counts:")
print("Class 0:", class_counts[0])
print("Class 1:", class_counts[1])

print("\nCalculated class weights:")
print("Class 0:", class_weights[0])
print("Class 1:", class_weights[1])

Training class counts:
Class 0: 152834
Class 1: 24742

Calculated class weights:
Class 0: 0.5809440307784917
Class 1: 3.5885538760003235


In [ ]:
import torch
import torch.nn as nn

# Convert the positive-class weight to a PyTorch tensor
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

# Weighted binary cross-entropy loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("Weighted loss function created successfully.")
print("Positive-class weight:", pos_weight.item())
print("Loss function:", criterion)

Weighted loss function created successfully.
Positive-class weight: 3.5885539054870605
Loss function: BCEWithLogitsLoss()


In [ ]:
import torch.optim as optim

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", 1e-3)

Optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Create DataLoaders
batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DataLoaders created successfully.")

print("\nBatch size:", batch_size)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully.

Batch size: 256
Training batches: 694
Validation batches: 149
Test batches: 149


In [ ]:
# Inspect one batch from the training DataLoader

X_batch, y_batch = next(iter(train_loader))

print("ONE TRAINING BATCH")
print("=" * 60)

print("Feature batch shape:", X_batch.shape)
print("Feature data type:", X_batch.dtype)

print("\nLabel batch shape:", y_batch.shape)
print("Label data type:", y_batch.dtype)

print("\nFirst patient's features:")
print(X_batch[0])

print("\nFirst 10 labels:")
print(y_batch[:10])

ONE TRAINING BATCH
Feature batch shape: torch.Size([256, 21])
Feature data type: torch.float32

Label batch shape: torch.Size([256])
Label data type: torch.float32

First patient's features:
tensor([ 0.0000,  0.0000,  1.0000,  0.2457,  0.0000,  0.0000,  0.0000,  1.0000,
         0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  1.0000, -0.4299, -0.4867,
         0.0000,  0.0000, -0.9939,  5.0000,  5.0000])

First 10 labels:
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 1.])


In [ ]:
# Put the model in training mode
model.train()

# Perform one forward pass
logits = model(X_batch)

print("FORWARD PASS")
print("=" * 60)

print("Input shape:", X_batch.shape)
print("Output shape:", logits.shape)

print("\nFirst 10 raw logits:")
print(logits[:10].squeeze())

# Convert logits to probabilities for inspection
probabilities = torch.sigmoid(logits)

print("\nFirst 10 probabilities:")
print(probabilities[:10].squeeze())

FORWARD PASS
Input shape: torch.Size([256, 21])
Output shape: torch.Size([256, 1])

First 10 raw logits:
tensor([0.2209, 0.1729, 0.1164, 0.0587, 0.3067, 0.1663, 0.2161, 0.3885, 0.2362,
        0.1253], grad_fn=<SqueezeBackward0>)

First 10 probabilities:
tensor([0.5550, 0.5431, 0.5291, 0.5147, 0.5761, 0.5415, 0.5538, 0.5959, 0.5588,
        0.5313], grad_fn=<SqueezeBackward0>)


In [ ]:
# Calculate the initial loss for the current batch

# Remove the final dimension from logits
# [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate weighted binary cross-entropy loss
initial_loss = criterion(batch_logits, y_batch)

print("INITIAL LOSS")
print("=" * 60)
print("Loss:", initial_loss.item())

INITIAL LOSS
Loss: 0.9943531155586243


In [ ]:
# Perform one complete training step

# Make sure the model is in training mode
model.train()

# Clear gradients from any previous step
optimizer.zero_grad()

# Forward pass
logits = model(X_batch)

# Remove the final dimension: [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate loss
loss = criterion(batch_logits, y_batch)

# Backpropagation
loss.backward()

# Update model parameters
optimizer.step()

print("ONE TRAINING STEP COMPLETED")
print("=" * 60)
print("Loss before parameter update:", loss.item())

ONE TRAINING STEP COMPLETED
Loss before parameter update: 1.0062642097473145


In [ ]:
# Check whether the model parameters were updated

print("PARAMETER UPDATE CHECK")
print("=" * 60)

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(
            f"{name:30s} | "
            f"mean = {parameter.data.mean().item():.6f} | "
            f"grad mean = {parameter.grad.mean().item():.6f}"
        )

PARAMETER UPDATE CHECK
network.0.weight               | mean = 0.003180 | grad mean = -0.000006
network.0.bias                 | mean = 0.006640 | grad mean = 0.000003
network.3.weight               | mean = 0.000014 | grad mean = 0.000203
network.3.bias                 | mean = -0.005297 | grad mean = 0.000374
network.6.weight               | mean = -0.001565 | grad mean = 0.001428
network.6.bias                 | mean = -0.013027 | grad mean = 0.004714
network.9.weight               | mean = 0.014504 | grad mean = 0.033711
network.9.bias                 | mean = 0.145329 | grad mean = 0.275640


In [ ]:
# Reset the model before real training

model = DiabetesDNN()

# Recreate the weighted loss
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Recreate the optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Model reset successfully.")
print("\nArchitecture:")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

print("\nOptimizer:")
print(optimizer.__class__.__name__)

print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Positive-class weight:", pos_weight.item())

Model reset successfully.

Architecture:
DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters:
13185

Optimizer:
Adam
Learning rate: 0.001
Positive-class weight: 3.5885539054870605
